# Domain 3 - Equity analysis rebuilt at local-government (LGA) level
**Update: the equity-deprivation index, previously state-level, rebuilt on LGA-level covariates.**

Composite deprivation index (0-100) from four equal-weighted components - remoteness (Weiss travel
time), low education, poverty, and low wealth (Meta Relative Wealth Index) - grouped into four
quartile tiers (Tier 1 = most deprived). Validated against the modelled LGA zero-dose rate, and
extended with multiscale geographically weighted regression (MGWR) to show place-varying drivers.

Prioritization principle (answer to the ED): rank LGAs **worst-to-best by modelled zero-dose burden**
(number of children), carrying the equity tier and archetype as columns - not by the index itself.

In [ ]:
!pip -q install pandas numpy scikit-learn mgwr libpysal geopandas matplotlib

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd

## 1. Load the LGA covariate/archetype master (includes modelled zero-dose)

In [ ]:
# From the archetype pipeline: LGA covariates + modelled zero-dose per LGA.
cov = pd.read_csv("lga_archetype_master.csv")
print(cov.shape); cov.filter(regex="travel|edu|poverty|wealth|zero_dose").head()

## 2. Composite equity-deprivation index (four equal-weighted components) and quartile tiers

In [ ]:
def mm(s):
    s = pd.to_numeric(s, errors="coerce"); return (s - s.min())/(s.max()-s.min())*100
comp = pd.DataFrame({
    "remoteness":    mm(cov["travel_time_hc"]),
    "low_education": mm(-cov["edu_mean_years_women_15_49"]),
    "poverty":       mm(cov["poverty_rate"]),
    "low_wealth":    mm(-cov["relative_wealth_index"]),
})
cov["equity_deprivation_index"] = comp.mean(axis=1).round(1)
cov["equity_tier"] = pd.qcut(cov["equity_deprivation_index"].rank(method="first"), 4,
                             labels=[4,3,2,1]).astype(int)   # 1 = most deprived
v = cov.dropna(subset=["zero_dose_rate_pct"])
print("Pearson r (index vs modelled zero-dose rate):",
      round(np.corrcoef(v["equity_deprivation_index"], v["zero_dose_rate_pct"])[0,1], 2))
cov["equity_tier"].value_counts().sort_index()

## 3. Prioritization - rank LGAs worst-to-best by modelled zero-dose burden

In [ ]:
rank = (cov.dropna(subset=["zero_dose_children"])
          .sort_values("zero_dose_children", ascending=False).reset_index(drop=True))
rank["burden_rank"] = np.arange(1, len(rank)+1)
tot = rank["zero_dose_children"].sum()
rank["cumulative_share_%"] = (rank["zero_dose_children"].cumsum()/tot*100).round(1)
cols = ["burden_rank","platform_State","platform_LGA","zero_dose_children","zero_dose_rate_pct",
        "equity_deprivation_index","equity_tier","archetype_type","cumulative_share_%"]
cols = [c for c in cols if c in rank.columns]
print("top 150 reach %.0f%%; top 270 reach %.0f%%" %
      (rank.loc[149,"cumulative_share_%"] if len(rank)>149 else 100,
       rank.loc[269,"cumulative_share_%"] if len(rank)>269 else 100))
rank[cols].head(15)

In [ ]:
rank[cols].to_csv("D3_lga_equity_burden_ranking.csv", index=False); print("saved worst-to-best ranking")

## 4. MGWR - place-varying drivers (optional, heavier)
Multiscale geographically weighted regression lets each driver's effect vary by place. On this
data OLS R2 ~ 0.39 rises to MGWR R2 ~ 0.55 across the modelled LGAs. Drivers: poverty, education,
travel time, wealth, stunting, conflict fatalities. White areas on the map = the 44 local
governments excluded for missing Penta1 (not modelled) - "no estimate", not "zero effect".
The full MGWR + geojson code is in `Archtyping at LGA level/domain3_mgwr.py`.

## 5. Interpretation
- Tier 1/Tier 2 deprivation concentrates in the North-East and North-West.
- Use the **burden ranking** as the spine (which LGA first) with the **equity tier + archetype** as
  the "why / what to do" columns; flag high-burden + high-deprivation LGAs as top priority.
- 730 of 774 LGAs calibrated; 44 excluded for missing Penta1 (documented).